In [2]:
import ast
import pandas as pd
import glob
import json
import numpy as np
import tqdm
from langchain_text_splitters import MarkdownHeaderTextSplitter
import os

import sys
sys.path.append("../..")
from benchmark.src import create_sentence_nace_code_similarities

In [3]:
all_results_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/similarity_search_descriptions/"

In [4]:
df_overview = pd.read_csv("../../data/datasets/stoxx_600/stoxx_600_overview.csv", sep=";")
df_overview = df_overview.dropna(subset="Report")
#df_overview = df_overview.dropna(subset="description_page")

In [5]:
report_jsons = glob.glob("../../data/datasets/stoxx_600/JSONs/*.json")
report_jsons.sort()
report_jsons

['../../data/datasets/stoxx_600/JSONs/AAK AB1.json',
 '../../data/datasets/stoxx_600/JSONs/ABB Ltd.2.json',
 '../../data/datasets/stoxx_600/JSONs/ANDRITZ AG1.json',
 '../../data/datasets/stoxx_600/JSONs/ASM International N.V.1.json',
 '../../data/datasets/stoxx_600/JSONs/ASR Nederland N.V.1.json',
 '../../data/datasets/stoxx_600/JSONs/AXA SA1.json',
 '../../data/datasets/stoxx_600/JSONs/Aalberts N.V.1.json',
 '../../data/datasets/stoxx_600/JSONs/Accelleron Industries AG1.json',
 '../../data/datasets/stoxx_600/JSONs/Acciona SA2.json',
 '../../data/datasets/stoxx_600/JSONs/Accor SA1.json',
 '../../data/datasets/stoxx_600/JSONs/Ackermans & van Haaren NV1.json',
 '../../data/datasets/stoxx_600/JSONs/Adecco Group AG1.json',
 '../../data/datasets/stoxx_600/JSONs/Admiral Group plc1.json',
 '../../data/datasets/stoxx_600/JSONs/Airbus SE1.json',
 '../../data/datasets/stoxx_600/JSONs/Akzo Nobel N.V.2.json',
 '../../data/datasets/stoxx_600/JSONs/Alcon AG1.json',
 '../../data/datasets/stoxx_600/JS

### Try to find table of contents

In [ ]:
os.environ["OPENAI_API_KEY"] = "your api key"

In [9]:
from langchain_ollama import OllamaLLM
from langchain.chat_models import ChatOpenAI

def query_ollama(prompt: str, model: str = "llama3.2:1b") -> str:
    """
    Simple function to call an Ollama model through LangChain.

    Args:
        prompt (str): The input text prompt for the model.
        model (str): The name of the Ollama model to use (default: "llama3").

    Returns:
        str: The model's generated response.
    """
    llm = OllamaLLM(model=model)
    response = llm.invoke(prompt)
    return response

def query_chatgpt(prompt: str, model: str = "gpt-4") -> str:
    """
    Simple function to call ChatGPT through LangChain.

    Args:
        prompt (str): The input text prompt for the model.
        model (str): The name of the ChatGPT model to use (default: "gpt-4").

    Returns:
        str: The model's generated response.
    """
    chat_model = ChatOpenAI(model=model)
    response = chat_model.invoke(prompt)
    return response

In [60]:
filter_table_and_header = lambda x : "\n".join([line for line in  x.split("\n") if len(line) > 0 and (line[0] == "#" or line[0] == "|")])

pages = [filter_table_and_header(page["markdown"]) for page in text["pages"]]

pages

['',
 '',
 '',
 '## About ABB',
 '## Table of contents\n## 16 Value creation\n|   18 | Who we are                        |\n|------|-----------------------------------|\n|   20 | Divisions and Business areas      |\n|   22 | Building a deep footprint         |\n|   24 | Our purpose                       |\n|   28 | Our strategy and priorities       |\n|   30 | Our operating model - the ABB Way |\n## 34 Sustainability in practice',
 '## 88 Risk and opportunities\n## 104  Governance structure\n## 112  Statutory Report\n## 166  Financial Statements',
 '## Letter from the Chairman and the Managing Director\n## External environment\n## Megatrends and local impact\n## Strategy for consistent performance',
 '## Sustainability in practice\n## A stable foundation for future growth',
 '',
 '## Key figures at a glance and five-year summary\n|                                                   |        |        |        | ( ` in Crores)   | ( ` in Crores)   |\n|-------------------------------------

In [61]:
table_of_contents_prompt = f"You are an analyst of an annual report: Which of the following pages does contain the table of contents?\n {pages[:10]}\n Only return the page number."

In [62]:
result = query_ollama(table_of_contents_prompt)

In [63]:
result

'The table of contents is located on pages 16 and 34.'

In [68]:
pages

['',
 '',
 '',
 '## About ABB',
 '## Table of contents\n## 16 Value creation\n|   18 | Who we are                        |\n|------|-----------------------------------|\n|   20 | Divisions and Business areas      |\n|   22 | Building a deep footprint         |\n|   24 | Our purpose                       |\n|   28 | Our strategy and priorities       |\n|   30 | Our operating model - the ABB Way |\n## 34 Sustainability in practice',
 '## 88 Risk and opportunities\n## 104  Governance structure\n## 112  Statutory Report\n## 166  Financial Statements',
 '## Letter from the Chairman and the Managing Director\n## External environment\n## Megatrends and local impact\n## Strategy for consistent performance',
 '## Sustainability in practice\n## A stable foundation for future growth',
 '',
 '## Key figures at a glance and five-year summary\n|                                                   |        |        |        | ( ` in Crores)   | ( ` in Crores)   |\n|-------------------------------------

In [76]:
for page in pages[:10]: 
    if page != "": 
        
        is_toc_on_page_prompt = f"""You are a precise and detail-oriented financial analyst specialized in reading and interpreting corporate annual reports.

        Task:
        Determine whether the following page from an annual report contains a description of the company’s business segments — meaning explanations of what the company produces, sells, or earns revenue from.

        Input:
        {page}

        Output format:
        Respond only with one of the following words:
        - Yes — if the page includes a description of business segments, products, or revenue-generating activities.
        - No — if it does not.

        Additional notes:
        - Ignore sections like financial tables, risk factors, management introductions, or sustainability discussions unless they explicitly describe business activities or products.
        - Do not include any explanation or reasoning — output must be exactly one word: Yes or No."""
        

        answer = query_ollama(is_toc_on_page_prompt, "llama3:8b")
        print(answer)
        if "yes" in answer.strip().lower(): 
            print("Page :", page)
            break

No
No
No
No
No
No


In [20]:
page

'18 Who we are\n\n20\n\nDivisions and Business areas\n\n22 Building a deep footprint\n\n24\n\nOur purpose\n\n28\n\nOur strategy and priorities\n\n30\n\nOur operating model - the ABB Way\n\n<!-- image -->'

In [29]:
def get_comp_description_page(full_pages):
    for i, page in enumerate(full_pages): 
        if page != "": 
            prompt = f"""You are a precise and detail-oriented financial analyst specialized in reading and interpreting corporate annual reports.

                Task:
                Determine whether the following page from an annual report contains a description of the company’s business segments — meaning explanations of what the company produces, sells, or earns revenue from.

                Few-shot examples:
                Example 1:
                Page:
                "The Group operates in three business segments: Automotive, Motorcycles, and Financial Services. The Automotive segment covers all activities relating to the development, production, assembly, and sale of automobiles."
                Answer: Yes

                Example 2:
                Page:
                "The Supervisory Board held six meetings during the fiscal year and discussed topics including corporate governance and risk management."
                Answer: No

                Example 3:
                Page:
                "The company’s purpose is to design sustainable packaging solutions across multiple industries, focusing on circular materials and innovation."
                Answer: Yes

                Key-words: 
                - "The Companies Business Model"
                - "Our business areas"
                - "Fundamental Information about the Group"
                - "Overview over the Group"
                - "What we do"
                
                Now analyze the following page:

                {page}

                Output format:
                Respond only with one of the following words:
                - Yes — if the page includes a description of business segments, products, or revenue-generating activities.
                - No — if it does not.

                Additional notes:
                - Ignore sections like table of contents, management reports, letters to shareholders, financial statements, or sustainability discussions unless they explicitly describe business activities or products.
                - Do not provide reasoning or explanations — your output must be exactly one word: Yes or No."""
            #answer = query_ollama(is_toc_on_page_prompt, "llama3:8b")
            answer = query_chatgpt(prompt, "gpt-4o")
            if "yes" in answer.content.strip().lower(): 
                print(page)
                break
    return i

In [31]:
results = []
for report_json in report_jsons[:5]: 
    with open(report_json, "r") as f: 
        text = json.load(f)
    full_pages = [page["markdown"] for page in text["pages"]]
    page = get_comp_description_page(full_pages)
    company_name = os.path.basename(report_jsons[0]).replace(".json", "")[:-1]
    results.append({"name":company_name, "page":page})

## Contents

## This is AAK

| Making Better Happen ™                           | 3   |
|--------------------------------------------------|-----|
| A Scandinavian company with a global presence    | 4   |
| Our vision                                       | 5   |
| 2022 in brief                                    |     |
| Key events                                       | 6   |
| Key figures                                      | 7   |
| Message from the Chair and the CEO               | 8   |
| What we do                                       |     |
| AAK in the value chain                           | 10  |
| AAK - a Multi-oil Ingredient House               | 11  |
| A broad range of raw materials                   | 12  |
| A business model built on Making Better Happen ™ | 13  |
| Strategy and aspiration                          |     |
| Continuous development of our strategy           | 15  |
| Updated portfolio strategy                       | 16  |
| Strategic actions to real

In [ ]:
result

'AAK AB'